# R3maJ Kaggle Notebook v4

Kaggle T4×2 training setup for R3maJ with Google Drive replay/checkpoint restore, **automatic checkpoint backup to Google Drive**, and **256 games**.

Drive access uses a Google service account credential supplied through a Kaggle Secret. No interactive OAuth is required.


In [ ]:
# 1. Clone R3maJ
import os, subprocess
ROOT='/kaggle/working/R3maJ'
REPO='https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT,'.git')):
    subprocess.run(['git','clone','--depth','1',REPO,ROOT],check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'],check=False)
print(ROOT)


In [ ]:
# 2. Install Google Drive API client and prepare paths
!pip install -q google-api-python-client google-auth google-auth-httplib2

import os, json, shutil, io, time, threading
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

DRIVE_ROOT_ID='1ktqAHT6wwYCyyRA4REBN2kZyFqeCbvKh'
LOCAL_ROOT='/kaggle/working/R3maJ/build'
LOCAL_REPLAY=f'{LOCAL_ROOT}/serialized_replays.bin'
LOCAL_CHECKPOINTS=f'{LOCAL_ROOT}/checkpoints'
os.makedirs(LOCAL_ROOT,exist_ok=True)

print('Drive root:',DRIVE_ROOT_ID)


In [ ]:
# 3. Authenticate with the Google service account and restore Drive data

# Kaggle Secret required: GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON
# Its value must be the complete service-account JSON.
# Share the R3maJ Drive folder with the service-account email as Editor.

from kaggle_secrets import UserSecretsClient

try:
    secret_client=UserSecretsClient()
    service_account_info=json.loads(secret_client.get_secret('GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON'))
except Exception as e:
    raise RuntimeError(
        "Missing Kaggle Secret GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON. Add the complete Google service-account JSON as a Kaggle Secret."
    ) from e

SCOPES=['https://www.googleapis.com/auth/drive']
creds=service_account.Credentials.from_service_account_info(service_account_info,scopes=SCOPES)
drive=build('drive','v3',credentials=creds,cache_discovery=False)

def drive_find_children(parent_id,name=None,mime_type=None):
    q=f"'{parent_id}' in parents and trashed = false"
    if name is not None:
        safe_name=name.replace("'","\\'")
        q += f" and name = '{safe_name}'"
    if mime_type:
        q += f" and mimeType = '{mime_type}'"
    return drive.files().list(q=q,fields='files(id,name,mimeType,size,modifiedTime)',pageSize=1000).execute().get('files',[])

def drive_download_file(file_id,dest):
    request=drive.files().get_media(fileId=file_id)
    with open(dest,'wb') as fh:
        downloader=MediaIoBaseDownload(fh,request,chunksize=16*1024*1024)
        done=False
        while not done:
            _,done=downloader.next_chunk()

def drive_download_tree(drive_folder_id,local_dir):
    os.makedirs(local_dir,exist_ok=True)
    for item in drive_find_children(drive_folder_id):
        path=os.path.join(local_dir,item['name'])
        if item['mimeType']=='application/vnd.google-apps.folder':
            drive_download_tree(item['id'],path)
        else:
            drive_download_file(item['id'],path)

print('Drive authentication: OK')
print('Service account:',service_account_info.get('client_email','<unknown>'))

DOWNLOAD_DIR='/kaggle/working/R3maJ_drive'
if os.path.exists(DOWNLOAD_DIR):
    shutil.rmtree(DOWNLOAD_DIR)
drive_download_tree(DRIVE_ROOT_ID,DOWNLOAD_DIR)

def find_file(base,name):
    for root,dirs,files in os.walk(base):
        if name in files: return os.path.join(root,name)
    return None
def find_dir(base,name):
    for root,dirs,files in os.walk(base):
        if name in dirs: return os.path.join(root,name)
    return None
src_replay=find_file(DOWNLOAD_DIR,'serialized_replays.bin')
src_checkpoints=find_dir(DOWNLOAD_DIR,'checkpoints')
assert src_replay,'serialized_replays.bin not found in shared R3maJ Drive folder.'
assert src_checkpoints,'checkpoints folder not found in shared R3maJ Drive folder.'
shutil.copy2(src_replay,LOCAL_REPLAY)
if os.path.exists(LOCAL_CHECKPOINTS): shutil.rmtree(LOCAL_CHECKPOINTS)
shutil.copytree(src_checkpoints,LOCAL_CHECKPOINTS)
print('Replay:',LOCAL_REPLAY)
print('Replay GB:',round(os.path.getsize(LOCAL_REPLAY)/(1024**3),3))
print('Checkpoint entries:',sorted(os.listdir(LOCAL_CHECKPOINTS))[:20])


### Drive layout

`R3maJ/serialized_replays.bin`
`R3maJ/checkpoints/<checkpoint directories>`

The authenticated Drive API restores the complete folder and later mirrors checkpoint changes back to the same `checkpoints/` folder.


In [ ]:
# 4. Install build dependencies + inspect both GPUs
import subprocess,os
subprocess.run(['apt-get','update','-qq'],check=False)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','git','libpython3-dev','pkg-config'],check=False)
import torch
print('torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),torch.version.cuda)
print('GPU count:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
assert torch.cuda.device_count() >= 2,'Expected Kaggle T4×2 runtime.'
subprocess.run(['nvidia-smi'],check=False)


In [ ]:
# 5. Configure + build
import os,subprocess,torch
os.chdir(ROOT)
prefix=os.path.dirname(torch.__file__)
subprocess.run(['cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release',f'-DTORCH_INSTALL_PREFIX={prefix}'],check=True)
subprocess.run(['cmake','--build','build','-j',str(os.cpu_count() or 2)],check=True)
EXE=os.path.join(ROOT,'build','R3maJ')
print('Binary:',EXE,os.path.exists(EXE))


In [ ]:
# 6. Verify restored data
assert os.path.exists(EXE),'R3maJ binary missing.'
assert os.path.exists(LOCAL_REPLAY),'Replay missing.'
assert os.path.isdir(LOCAL_CHECKPOINTS),'Checkpoint directory missing.'
print('READY')
print('Games:',256)
print('Checkpoint entries:',len([x for x in os.listdir(LOCAL_CHECKPOINTS) if not x.startswith('.')]))


## Google Drive checkpoint backup

v4 now uses the **Google Drive API with a service account** for both restore and Kaggle → Drive checkpoint backup.

### One-time setup

1. Create a Google Cloud service account in the same project.
2. Create/download its JSON key.
3. Share the `R3maJ` Drive folder with the service-account email as **Editor**.
4. In Kaggle Secrets, create `GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON` and paste the complete JSON key as its value.

The watcher checks the local checkpoint tree every **60 seconds** and synchronizes completed/changed files to Drive. No interactive Google login is needed.


In [ ]:
# 7. Prepare the Drive checkpoint folder
SYNC_INTERVAL=60
_stop_backup=False
_last_uploaded={}

checkpoint_folders=drive_find_children(DRIVE_ROOT_ID,name='checkpoints',mime_type='application/vnd.google-apps.folder')
if checkpoint_folders:
    DRIVE_CHECKPOINTS_ID=checkpoint_folders[0]['id']
else:
    meta={'name':'checkpoints','mimeType':'application/vnd.google-apps.folder','parents':[DRIVE_ROOT_ID]}
    DRIVE_CHECKPOINTS_ID=drive.files().create(body=meta,fields='id').execute()['id']

print('Drive checkpoint folder ID:',DRIVE_CHECKPOINTS_ID)


In [ ]:
# 8. Background checkpoint watcher: Kaggle -> Google Drive

def ensure_drive_folder(name,parent_id):
    found=drive_find_children(parent_id,name=name,mime_type='application/vnd.google-apps.folder')
    if found: return found[0]['id']
    meta={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]}
    return drive.files().create(body=meta,fields='id').execute()['id']

def upload_file_to_drive(local_path,parent_id,name):
    found=drive_find_children(parent_id,name=name)
    media=MediaFileUpload(local_path,resumable=True)
    if found:
        drive.files().update(fileId=found[0]['id'],media_body=media).execute()
    else:
        meta={'name':name,'parents':[parent_id]}
        drive.files().create(body=meta,media_body=media,fields='id').execute()

def sync_checkpoint_tree():
    if not os.path.isdir(LOCAL_CHECKPOINTS): return
    for root,dirs,files in os.walk(LOCAL_CHECKPOINTS):
        rel_root=os.path.relpath(root,LOCAL_CHECKPOINTS)
        parent_id=DRIVE_CHECKPOINTS_ID
        if rel_root != '.':
            for part in rel_root.split(os.sep):
                parent_id=ensure_drive_folder(part,parent_id)
        for name in files:
            local_path=os.path.join(root,name)
            try:
                sig=(os.path.getsize(local_path),os.path.getmtime_ns(local_path))
                key=os.path.relpath(local_path,LOCAL_CHECKPOINTS)
                if _last_uploaded.get(key)==sig: continue
                time.sleep(0.05)
                if sig != (os.path.getsize(local_path),os.path.getmtime_ns(local_path)): continue
                upload_file_to_drive(local_path,parent_id,name)
                _last_uploaded[key]=sig
                print(f'[Drive backup] uploaded: {key}')
            except FileNotFoundError:
                continue

def backup_loop():
    print(f'[Drive backup] watcher started; checking every {SYNC_INTERVAL}s')
    while not _stop_backup:
        try: sync_checkpoint_tree()
        except Exception as e: print('[Drive backup] ERROR:',repr(e))
        for _ in range(SYNC_INTERVAL):
            if _stop_backup: break
            time.sleep(1)

backup_thread=threading.Thread(target=backup_loop,daemon=True)
backup_thread.start()


In [ ]:
# 9. Start training — 256 games
import os,subprocess
os.chdir(os.path.join(ROOT,'build'))
TRAIN_ARGS=['--device','cuda','--save-dir','checkpoints','--games','256','--replays','serialized_replays.bin']
print('Launching:','./R3maJ',*TRAIN_ARGS)
proc=subprocess.Popen(['./R3maJ']+TRAIN_ARGS)
print('Training PID:',proc.pid)


## T4×2 note

Kaggle provides two Tesla T4 GPUs. The current R3maJ executable has no verified multi-GPU collector/learner option, so v4 keeps the safe single-process `--device cuda` configuration.

The important addition is **checkpoint persistence**: the authenticated service account lets the notebook restore the Drive tree at startup and continuously mirror completed checkpoints back to Drive.
